# Esercitazione 7

## Esercizio 1
### MongoDB

Dopo aver creato il vostro database sul cloud MongoDB caricate i database di esempio (andate in Deployment -> Database e fra le opzioni del Cluster scegliete "Load sample dataset")

Verificate clicckando su "Browse Collections" che i database di esempio siano stati caricati, userete il database sample_mflix.


Preparate il notebook per l'accesso al DB come visto in MongoDB_CRUD.ipynb

In [2]:
!pip install -q dnspython pymongo

In [3]:
import datetime
import pprint
import pymongo
from pymongo import MongoClient

Autorizzate l'IP della vostra macchina alla connessione al vostro DB (andate in Security -> Network access).

Andate quindi in Database -> Connect -> Drivers per ottenere la stringa di connessione e fate la connessione.

In [4]:
client = MongoClient("mongodb+srv://<username>:<password>@cluster0.mongodb.net/test?retryWrites=true&w=majority")
db = client.sample_airbnb

ConfigurationError: The DNS query name does not exist: _mongodb._tcp.cluster0.mongodb.net.

In [ ]:
# Verifichiamo che sia tutto ok accedendo alla collezione listingAndReview di sample_airbnb
list(db.listingsAndReviews.find().limit(1))

1. Trovare tutti gli alloggi di tipo 'House' in United States che possono ospitare 4 persone

In [ ]:
query = {"property_type" : "House", "address.country" : "United States", "accommodates":{"$gte":5}}
prop = {"_id":0, "name":1, "minimum_nights":1, "accommodates":1, "address.country":1}
list(db.listingsAndReviews.find(query, prop).limit(5))

2. Trovare gli alloggi che hanno un numero di review maggiore di 100

In [ ]:
prop = {"_id":0, "name":1, "number_of_reviews":1, "accommodates":1, "address.country":1}
res=db.listingsAndReviews.find ( {"number_of_reviews": {"$gt": 100 } }, prop)
list(res.limit(3))

3. Trovare gli alloggi che hanno fra 3 e 5 letti

In [ ]:
prop = {"_id":0, "name":1, "beds":1, "accommodates":1, "address.country":1}
res=db.listingsAndReviews.find ( {"beds": {"$gte": 3 , "$lte": 5} }, prop)
list(res.limit(3))

4. Trovare gli allogi che hanno fra le amenities "Wifi" e "Microwave"

In [ ]:
prop = {"_id":0, "name":1, "beds":1, "amenities":1, "address.country":1}
res=db.listingsAndReviews.find ( { "amenities": {"$all": ["Wifi", "Microwave"] }}, prop)
list(res.limit(3))

5. Trovare il numero totale di alloggi

In [ ]:
group_stg = {"$group":{"_id":"null", "count":{"$sum":1}}}
res = db.listingsAndReviews.aggregate([group_stg])
list(res)

6. Trovare il numero di alloggi in "United States"

In [ ]:
match_stg = {"$match" : {"address.country":"United States"}}
group_stg = {"$group":{"_id":"null", "count":{"$sum":1}}}
res = db.listingsAndReviews.aggregate([match_stg, group_stg])
list(res)

7. Trovare il numero di alloggi con più di 2 letti per ciascun paese

In [ ]:
match_stg = {"$match" : {"beds":{"$gt": 2}}}
group_stg = {"$group":{"_id":"$address.country", "count":{"$sum":1}}}
res = db.listingsAndReviews.aggregate([match_stg, group_stg])
list(res)

8. Trovare i 5 alloggi in United States col minor numero di amenities

In [ ]:
match_stg = {"$match" : {"address.country":"United States"}}
project_stg = {"$project" : {"_id":0, "name":1, "beds":1, "amenities":1, "address.country":1, "namenities": {"$size":"$amenities"}}}
sort_stg = {"$sort": {"namenities":1}}
limit_stg = {"$limit":5}
res = db.listingsAndReviews.aggregate([match_stg, project_stg, sort_stg, limit_stg])
list(res)

## Esercizio 2
### Cypher

Create la vostra istanza di Neo4J su AuraDB, annotate URI, user e password e importate il Movie database.

A questo punto potrete procedere con la connessione da Python al DB.

Preparate il notebook per l'accesso al DB come visto in Neo4j_CRUD.ipynb

In [ ]:
#!pip install neo4j

from neo4j import GraphDatabase

# Update with your Neo4j credentials and connection URI
uri = "bolt://54.197.28.170"
user = "neo4j"
password = "catches-glues-instructor"

driver = GraphDatabase.driver(uri, auth=(user, password))

Funzione di comodo per eseguire le query e visualizzare i risultati

In [ ]:
def run_query(query):
    with driver.session() as session:
        result = session.run(query)
        return [record.data() for record in result]

1. Movies released after 2000

In [ ]:
query1 = """
MATCH (m:Movie)
WHERE m.released > 2000
RETURN m.title AS title, m.released AS year
ORDER BY year
"""

run_query(query1)

2. Actors who acted in The Matrix

In [ ]:
query2 = """
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: "The Matrix"})
RETURN p.name AS actor
"""

run_query(query2)

3. People who directed and acted in movies

In [ ]:
query3 = """
MATCH (p:Person)-[:DIRECTED]->(m:Movie)
WHERE (p)-[:ACTED_IN]->()
RETURN p.name AS director_actor, m.title AS movie
"""

run_query(query3)

ServiceUnavailable: Couldn't connect to 54.197.28.170:7687 (resolved to ('54.197.28.170:7687',)):
Timed out trying to establish connection to ResolvedIPv4Address(('54.197.28.170', 7687))

4. Actor movie counts

In [ ]:
query4 = """
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
RETURN p.name AS actor, COUNT(m) AS movies
ORDER BY movies DESC
"""

run_query(query4)

5. Actor pairs who acted in the same movie

In [ ]:
query5 = """
MATCH (p1:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(p2:Person)
WHERE p1 <> p2
RETURN DISTINCT m.title AS movie, p1.name AS actor1, p2.name AS actor2
ORDER BY movie
"""

run_query(query5)

## Esercizio 3
### Simulazione Hadoop Streaming con le pipe di UNIX

Abbiamo a disposizione due dataset: il primo contiene i dati degli utenti, il secondo contiene i dati dei tweets.

Il dataset degli **utenti** è costituito da un unico file CSV.
Ciascuna riga è composta da due campi, separati da virgola: `user_id` e `screen_name`

Il dataset dei **tweets** è costituito da 8 file CSV. Ciascuna riga è composta da 6 campi, separati da virgola: `tweet_id`, `retweet_count`, `favorite_count`, `user_id`, `created_at` (data e ora in cui è stato inviato il tweet, nel formato `anno-mese-giorno` _spazio_ `ora:minuto:secondo`), `hashtags` (uno o più valori, separati da punto e virgola).

Scrivere mapper e reducer in python per:
1. Contare il numero di occorrenze di ciascun hashtag
2. Restituire i 25 hashtag più frequenti
3. Contare il numero di tweets inviati, per ogni fascia oraria:
   - dalle 06:00 alle 14:59 -> Mattina
   - dalle 15:00 alle 19:59 -> Pomeriggio
   - dalle 20:00 alle 23:59 e dalle 00:00 alle 05:59 -> Sera e notte

In [ ]:
# Conta numero di occorrenze di ciascun hashtag
# Copiamo nel nostro Google Drive i dati e gli script Python ed eseguiamo il comando shell
# che simula l'esecuzione di un job di Hadoop streaming.
# Il mapper produrrà coppie (hashtag, 1), come reducer possiamo quindi usare lo stesso del word count.
!cat tweeter/input_tweets/* | \
    python3 SolEs07_py/hashtag_count_mapper.py | \
    sort | \
    python3 SolEs07_py/hashtag_count_reducer.py > \
    hashtag_count.txt

In [ ]:
# Trova i 25 hashtag più frequenti
# Nota: possiamo usare come input l'output dell'analisi precedente che abbiamo salvato sul Drive
# E' sufficente un mapper che costruisce la lista dei 25 più frequenti. L'output del mapper è già
# il risultato cercato.
!cat hashtag_count.txt | \
    python3 SolEs07_py/hashtag_top25_mapper.py

In [ ]:
# Conta numero di tweet per fascia oraria
# Il mapper produce coppie chiave valore del tipo ("Mattina", 1), ("Pomeriggio", 1), etc...
# Come reducer possiamo quindi usare ancora lo stesso del word count
!cat tweeter/input_tweets/* | \
    python3 SolEs07_py/timeslot_count_mapper.py | \
    sort | \
    python3 SolEs07_py/timeslot_count_reducer.py

# Esercizio 4
### Analisi dati con Spark RDD
Il dataset `prodotti` contiene i dati dei prodotti in vendita:
- un identificativo univoco (`item_id`)
- la categoria del prodotto (`category`)
- il prezzo del singolo prodotto (`price`)

Il dataset `venditori` contiene i dati dei venditori:
- un identificativo univoco (`seller_id`)
- il tipo del venditore (`seller_type`), che può essere First-Party o Third-Party

Il dataset `transazioni` contiene i dati delle vendite effettuate:
- id univoco della transazione (`transaction_id`)
- id univoco del prodotto venduto (`item_id`)
- id univoco del venditore (`seller_id`)
- anno, mese e giorno della transazione (`year`, `month` e `day`)
- quantità di prodotti venduti (`num_items`)

In [5]:
import findspark
findspark.init()

#importing pyspark
import pyspark
#importing sparksession
from pyspark.sql import SparkSession

#creating a sparksession object and providing appName
spark=SparkSession.builder.appName("local").getOrCreate()
sc = spark.sparkContext

In [6]:
transazioni = spark.read.csv('vendite/transazioni.csv', header=True, inferSchema=True).rdd.cache()
prodotti = spark.read.csv('vendite/prodotti.csv', header=True, inferSchema=True).rdd.cache()
venditori = spark.read.csv('vendite/venditori.csv', header=True, inferSchema=True).rdd.cache()

In [7]:
transazioni.take(3)

[Row(transaction_id=1, item_id=1, seller_id=1, year=2009, month=5, day=29, num_items=1),
 Row(transaction_id=2, item_id=2, seller_id=8, year=2009, month=11, day=30, num_items=1),
 Row(transaction_id=3, item_id=3, seller_id=15, year=2009, month=1, day=27, num_items=1)]

In [8]:
prodotti.take(3)

[Row(item_id=1, category='Health & Personal Care', price=18.09),
 Row(item_id=2, category='Books', price=15.68),
 Row(item_id=3, category='Books', price=13.57)]

In [9]:
venditori.take(3)

[Row(seller_id=1, seller_type='Third-Party'),
 Row(seller_id=8, seller_type='Third-Party'),
 Row(seller_id=15, seller_type='First-Party')]

1. Per ogni mese, restituire la quantità venduta (`num_items`) massima in una singola transazione.

In [10]:
month_num_items = transazioni.map(lambda row: (row.month, row.num_items))
month_num_items.reduceByKey(max).sortByKey().collect()

[(1, 3),
 (2, 3),
 (3, 15),
 (4, 2),
 (5, 2),
 (6, 3),
 (7, 2),
 (8, 3),
 (9, 2),
 (10, 16),
 (11, 3),
 (12, 5)]


2. Calcolare il ricavo ottenuto da ogni transazione, moltiplicando il prezzo del prodotto e la quantità venduta.

In [11]:
# Creiamo i Pair RDD con item_id come chiave
item_id_prodotti = prodotti.map(lambda row: (row.item_id, row))
item_id_transazioni = transazioni.map(lambda row: (row.item_id, row))
# Effettuiamo il join e manteniamo solo i valori
prodotti_transazioni = item_id_prodotti.join(item_id_transazioni).values().cache()
# Calcoliamo il ricavo per ogni transazione
def calc_ricavi(prod_tr_row):
    prodotti_row, transazioni_row = prod_tr_row
    ricavo = prodotti_row.price * transazioni_row.num_items

    value = {'prodotto': prodotti_row, 'transazione': transazioni_row, 'ricavo': ricavo}
    return value
prod_tr_ricavi = prodotti_transazioni.map(calc_ricavi)
prod_tr_ricavi.take(3)

[{'prodotto': Row(item_id=2, category='Books', price=15.68),
  'transazione': Row(transaction_id=2, item_id=2, seller_id=8, year=2009, month=11, day=30, num_items=1),
  'ricavo': 15.68},
 {'prodotto': Row(item_id=4, category='Books', price=23.1),
  'transazione': Row(transaction_id=4, item_id=4, seller_id=13, year=2009, month=12, day=21, num_items=1),
  'ricavo': 23.1},
 {'prodotto': Row(item_id=6, category='Electronics', price=59.99),
  'transazione': Row(transaction_id=6, item_id=6, seller_id=7, year=2009, month=3, day=17, num_items=1),
  'ricavo': 59.99}]


3. Calcolare il ricavo massimo ottenuto da ciascun venditore.

In [12]:
# Creiamo un Pair RDD del tipo (seller_id, ricavo)
seller_id_ricavi = prod_tr_ricavi.map(lambda x: (x['transazione'].seller_id, x['ricavo']))
# Calcoliamo il ricavo massimo per ciascun venditore
seller_id_ricavi.reduceByKey(max).sortByKey().collect()

[(1, 119.99),
 (2, 180.0),
 (3, 214.18),
 (4, 130.0),
 (5, 107.09),
 (6, 107.09),
 (7, 107.09),
 (8, 107.09),
 (9, 829.0),
 (10, 829.0),
 (11, 179.99),
 (12, 147.38),
 (13, 359.0),
 (14, 107.09),
 (15, 359.0),
 (16, 109.99),
 (17, 214.18),
 (18, 242.99),
 (19, 106.31),
 (20, 299.99)]


4. Calcolare il ricavo totale ottenuto da ciascun venditore.

In [13]:
# Calcoliamo il ricavo totale per ciascun venditore
ricavi_totali_venditori = seller_id_ricavi.reduceByKey(lambda x, y: x + y)
# Ordiniamo le chiavi e arrotondiamo i valori
ricavi_totali_venditori.mapValues(lambda x: round(x, 2)).sortByKey().collect()

[(1, 1206.58),
 (2, 1289.63),
 (3, 1312.22),
 (4, 1213.1),
 (5, 812.48),
 (6, 1041.86),
 (7, 854.68),
 (8, 906.3),
 (9, 2263.64),
 (10, 1695.16),
 (11, 2702.78),
 (12, 2805.15),
 (13, 3598.16),
 (14, 2238.86),
 (15, 3477.58),
 (16, 2654.43),
 (17, 2816.42),
 (18, 2285.16),
 (19, 2640.56),
 (20, 2369.13)]


5. Calcolare il ricavo medio ottenuto da ciascun venditore.

In [14]:
# Trasformazioni per ordinare un generico Pair RDD: mapValues + reduceByKey + mapValues
def avg_by_key(pair_rdd):
    return pair_rdd.mapValues(lambda x: (x, 1)).reduceByKey(sum_values_counters).mapValues(calc_avg)

def sum_values_counters(x, y):
    value1, count1 = x
    value2, count2 = y
    return value1 + value2, count1 + count2

def calc_avg(sum_values_counter):
    sum_values, counter = sum_values_counter
    return sum_values / counter

ricavi_medi = avg_by_key(seller_id_ricavi).mapValues(lambda x: round(x, 2))
ricavi_medi.sortByKey().collect()

[(1, 24.62),
 (2, 29.99),
 (3, 26.78),
 (4, 26.37),
 (5, 21.96),
 (6, 23.68),
 (7, 18.58),
 (8, 22.66),
 (9, 51.45),
 (10, 42.38),
 (11, 25.26),
 (12, 31.52),
 (13, 34.93),
 (14, 25.73),
 (15, 35.13),
 (16, 27.94),
 (17, 29.96),
 (18, 30.47),
 (19, 24.23),
 (20, 26.62)]


6. Ordinare i risultati ottenuti nell'esercizio precedente in base al ricavo medio, in modo decrescente.

In [15]:
ricavi_medi.sortBy(lambda x: x[1], ascending=False).collect()

[(9, 51.45),
 (10, 42.38),
 (15, 35.13),
 (13, 34.93),
 (12, 31.52),
 (18, 30.47),
 (2, 29.99),
 (17, 29.96),
 (16, 27.94),
 (3, 26.78),
 (20, 26.62),
 (4, 26.37),
 (14, 25.73),
 (11, 25.26),
 (1, 24.62),
 (19, 24.23),
 (6, 23.68),
 (8, 22.66),
 (5, 21.96),
 (7, 18.58)]

7. Calcolare i 10 venditori che hanno ottenuto la somma di ricavi maggiore nel mese di Maggio del 2009.


In [16]:
# Filtriamo l'RDD per ottenere solo le transazioni di Maggio 2009
# Creiamo un Pair RDD del tipo (seller_id, ricavo)
# Calcoliamo la somma dei ricavi per ogni chiave (cioè per ogni venditore)
def get_maggio2019(x):
    return x['transazione'].month == 5 and x['transazione'].year == 2009

seller_ricavi_maggio = (prod_tr_ricavi
                            .filter(get_maggio2019)
                            .map(lambda x: (x['transazione'].seller_id, x['ricavo']))
                            .reduceByKey(lambda x, y: x + y))

# Utilizziamo top per ottenere i 10 venditori con la somma di ricavi maggiore
seller_ricavi_maggio.top(10, key=lambda x: x[1])

[(15, 632.76),
 (12, 425.61),
 (17, 382.45),
 (18, 315.5),
 (20, 303.53),
 (11, 300.33),
 (4, 280.78),
 (8, 186.44),
 (14, 166.25),
 (19, 157.83999999999997)]

8. Calcolare il ricavo medio ottenuto dai venditori "First-Party" per ogni categoria di prodotto.

In [25]:
# Le informazioni che ci servono sono le seguenti:
# `seller_type`, che si trova nell'RDD `venditori`
# `ricavo`, che si trova nell'RDD `prod_tr_ricavi`
# `category`, che si trova nell'RDD `prod_tr_ricavi`

print( prod_tr_ricavi.collect() )
# Filtriamo i venditori mantenendo solo quelli First-Party
venditori_first = venditori.filter(lambda row: row.seller_type == 'First-Party')

print( venditori_first.collect() )
# Facciamo il join con `prod_tr_ricavi` in base al `seller_id`
seller_id_venditori_first = venditori_first.map(lambda row: (row.seller_id, row))
seller_id_prod_tr_ricavi = prod_tr_ricavi.map(lambda x: (x['transazione'].seller_id, x))
print( seller_id_prod_tr_ricavi.collect() )
seller_prod_tr_ricavi = seller_id_venditori_first.join(seller_id_prod_tr_ricavi).values().cache()

print( seller_prod_tr_ricavi.collect() )
# Creiamo il pair rdd `(categoria, ricavo)`
categoria_ricavo = seller_prod_tr_ricavi.values().map(lambda x: (x['prodotto'].category, x['ricavo']))
# Calcoliamo la media per ogni categoria
(avg_by_key(categoria_ricavo)               # calcoliamo la media
     .mapValues(lambda x: round(x, 2))      # arrotondiamo i valori
     .sortByKey()                           # ordiniamo in base alla chiave
     .collectAsMap())                       # otteniamo i risultati come dizionario

[{'prodotto': Row(item_id=2, category='Books', price=15.68), 'transazione': Row(transaction_id=2, item_id=2, seller_id=8, year=2009, month=11, day=30, num_items=1), 'ricavo': 15.68}, {'prodotto': Row(item_id=4, category='Books', price=23.1), 'transazione': Row(transaction_id=4, item_id=4, seller_id=13, year=2009, month=12, day=21, num_items=1), 'ricavo': 23.1}, {'prodotto': Row(item_id=6, category='Electronics', price=59.99), 'transazione': Row(transaction_id=6, item_id=6, seller_id=7, year=2009, month=3, day=17, num_items=1), 'ricavo': 59.99}, {'prodotto': Row(item_id=8, category='Electronics', price=3.33), 'transazione': Row(transaction_id=8, item_id=8, seller_id=5, year=2009, month=7, day=3, num_items=2), 'ricavo': 6.66}, {'prodotto': Row(item_id=10, category='Books', price=11.97), 'transazione': Row(transaction_id=10, item_id=10, seller_id=5, year=2009, month=11, day=30, num_items=1), 'ricavo': 11.97}, {'prodotto': Row(item_id=12, category='Kitchen & Housewares', price=2.29), 'tran

{'Apparel & Accessories': 31.99,
 'Automotive': 9.37,
 'Baby': 29.98,
 'Books': 30.14,
 'DVD': 19.73,
 'Electronics': 37.17,
 'Health & Personal Care': 31.65,
 'Health & Personal Care Appliances': 22.49,
 'Jewelry': 37.0,
 'Kindle Hardware': 218.0,
 'Kindle eBooks': 6.68,
 'Kitchen & Housewares': 41.64,
 'MP3 Downloads': 3.9,
 'Magazine Subscriptions': 19.95,
 'Music': 19.92,
 'Musical Instruments': 84.75,
 'Office Products': 19.49,
 'Other': 1.99,
 'Pet Supplies': 20.74,
 'Software': 271.49,
 'Sports & Outdoors': 13.2,
 'Tools & Hardware': 28.75,
 'Toys & Games': 14.87,
 'Video Games': 34.49,
 'Video On Demand Videos': 1.99}

# Esercizio 5
### Ripetere le analisi dell'esercizio 4 utilizzando SparkSQL

In [18]:
spark.read.csv('vendite/transazioni.csv', header=True, inferSchema=True).createOrReplaceTempView('transazioni')
spark.read.csv('vendite/prodotti.csv', header=True, inferSchema=True).createOrReplaceTempView('prodotti')
spark.read.csv('vendite/venditori.csv', header=True, inferSchema=True).createOrReplaceTempView('venditori')

In [19]:
spark.sql('''
SELECT * FROM transazioni
''').show(3)

+--------------+-------+---------+----+-----+---+---------+
|transaction_id|item_id|seller_id|year|month|day|num_items|
+--------------+-------+---------+----+-----+---+---------+
|             1|      1|        1|2009|    5| 29|        1|
|             2|      2|        8|2009|   11| 30|        1|
|             3|      3|       15|2009|    1| 27|        1|
+--------------+-------+---------+----+-----+---+---------+
only showing top 3 rows


In [20]:
spark.sql('''
SELECT * FROM prodotti
''').show(3)

+-------+--------------------+-----+
|item_id|            category|price|
+-------+--------------------+-----+
|      1|Health & Personal...|18.09|
|      2|               Books|15.68|
|      3|               Books|13.57|
+-------+--------------------+-----+
only showing top 3 rows


In [21]:
spark.sql('''
SELECT * FROM venditori
''').show(3)

+---------+-----------+
|seller_id|seller_type|
+---------+-----------+
|        1|Third-Party|
|        8|Third-Party|
|       15|First-Party|
+---------+-----------+
only showing top 3 rows


1. Per ogni mese, restituire la quantità venduta (`num_items`) massima in una singola transazione.

In [22]:
spark.sql('''
SELECT month, max(num_items) AS max_num_items
FROM transazioni
GROUP BY month
ORDER BY month
''').show()

+-----+-------------+
|month|max_num_items|
+-----+-------------+
|    1|            3|
|    2|            3|
|    3|           15|
|    4|            2|
|    5|            2|
|    6|            3|
|    7|            2|
|    8|            3|
|    9|            2|
|   10|           16|
|   11|            3|
|   12|            5|
+-----+-------------+




2. Calcolare il ricavo ottenuto da ogni transazione, moltiplicando il prezzo del prodotto e la quantità venduta.

...